# Model Design and Decisions

The modeling layer estimates relative species use and environmental plausibility from engineered H3/date features. This section explains the response variable, model families, validation choices, and the reasons those choices matter for interpreting the final risk products.


## Response Variable

Telemetry-derived species use is represented by a residence index:

$$
R(h,t,s)=C(h,t,s)\times I(h,t,s)
$$

where `C` is telemetry record count and `I` is tracked individual count for H3 cell `h`, date `t`, and species `s`. The modeling target is log transformed:

$$
Y(h,t,s)=\log(1+R(h,t,s))
$$

## Joint-Species Model

The workflow uses a joint-species modeling approach. Species identity is encoded as categorical indicators appended to the numerical predictor matrix. This lets one model learn shared environmental structure while preserving species-specific responses.

## Learner and Validation Decisions

The workflow evaluated histogram gradient boosting, random forest, Extra Trees, and a Bayesian/GMM-style candidate during learner screening. Extra Trees was selected as the primary species-use learner.

Row-random validation was treated as an initial benchmark only because randomly mixed rows can overstate transferability. The final validation framing used SOM-hierarchical k=30 seascape classes as environmental groups in five-fold grouped cross-validation. Those seascape group labels are read from `data/modeling/environmental_regimes/`, the consolidated H3/date table that also stores Bayesian GMM environmental components.

## Balancing and Weights

Because zero-use rows greatly outnumber positive-use rows, all positive rows were retained and an equal number of zero-use rows was sampled. Sample weights increased the influence of higher-use observations:

$$
w(h,t,s)=1+R(h,t,s)^{0.75}
$$

## Plausibility Layer Decision

The Bayesian/GMM-style candidate was not selected as the primary species-use learner. Its environmental-density component was retained as the plausibility layer, which diagnoses whether predictions are made under environmental conditions similar to observed telemetry-use conditions. This is an environmental-support diagnostic, not a species-presence probability.